INSTALLATION AND PREPROCESSING

In [9]:
#Cell 1: Installations
!pip install mne scikit-learn pandas pygame

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Cell 2: Inspect the channel names to avoid errors
import mne
import os

print("Inspecting channel names from one file...")
# Path to one of the files we downloaded
file_to_inspect = "eeg-during-mental-arithmetic-tasks-1.0.0\Subject00_1.edf"

# Load the file just to read its info
raw_inspect = mne.io.read_raw_edf(file_to_inspect, verbose=False)

# Print all available channel names
print("✅ Available channel names are:")
print(raw_inspect.ch_names)

<>:7: SyntaxWarning: invalid escape sequence '\S'
<>:7: SyntaxWarning: invalid escape sequence '\S'
C:\Users\BIT\AppData\Local\Temp\ipykernel_18924\1569482754.py:7: SyntaxWarning: invalid escape sequence '\S'
  file_to_inspect = "eeg-during-mental-arithmetic-tasks-1.0.0\Subject00_1.edf"


Inspecting channel names from one file...
✅ Available channel names are:
['EEG Fp1', 'EEG Fp2', 'EEG F3', 'EEG F4', 'EEG F7', 'EEG F8', 'EEG T3', 'EEG T4', 'EEG C3', 'EEG C4', 'EEG T5', 'EEG T6', 'EEG P3', 'EEG P4', 'EEG O1', 'EEG O2', 'EEG Fz', 'EEG Cz', 'EEG Pz', 'EEG A2-A1', 'ECG ECG']


In [11]:
# Cell 3: Load and Process Data for Multiple Subjects (Corrected with Epoch Preloading)
import mne
import os
import numpy as np

# --- Reusable function to process one file ---
def preprocess_data(file_path):
    """Loads an EEG file, selects the 'EEG Fz' channel, filters, and creates/preloads epochs."""
    if not os.path.exists(file_path):
        print(f"  Warning: File not found at {file_path}")
        return None
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    correct_channel_name = 'EEG Fz'
    raw.pick(correct_channel_name)
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    
    # --- THIS IS THE CORRECTED LINE ---
    # We add preload=True to load the epoch data into memory immediately.
    epochs = mne.make_fixed_length_epochs(raw, duration=2, overlap=1, preload=True, verbose=False)
    
    return epochs

# --- Main loop to process subjects 1 through 5 ---
subjects_to_process = [f"Subject{i:02d}" for i in range(1, 6)]
data_folder = "eeg-during-mental-arithmetic-tasks-1.0.0"

all_low_attention_epochs = []
all_high_attention_epochs = []

print(f"\nStarting data processing for subjects: {subjects_to_process}...")

for subject in subjects_to_process:
    print(f"  Processing {subject}...")
    low_attention_file = os.path.join(data_folder, f"{subject}_1.edf")
    high_attention_file = os.path.join(data_folder, f"{subject}_2.edf")

    # Process the low attention file
    low_epochs = preprocess_data(low_attention_file)
    if low_epochs:
        all_low_attention_epochs.append(low_epochs)

    # Process the high attention file
    high_epochs = preprocess_data(high_attention_file)
    if high_epochs:
        all_high_attention_epochs.append(high_epochs)

# Check if lists are empty before trying to concatenate
if not all_low_attention_epochs or not all_high_attention_epochs:
    print("\nError: Could not load any epoch data. Please check file paths and names.")
else:
    # Combine the epochs from all subjects into single lists
    final_low_epochs = mne.concatenate_epochs(all_low_attention_epochs)
    final_high_epochs = mne.concatenate_epochs(all_high_attention_epochs)
    print("\n✅ Processing complete!")
    print(f"Total low-attention epochs from all subjects: {len(final_low_epochs)}")
    print(f"Total high-attention epochs from all subjects: {len(final_high_epochs)}")


Starting data processing for subjects: ['Subject01', 'Subject02', 'Subject03', 'Subject04', 'Subject05']...
  Processing Subject01...
  Processing Subject02...
  Processing Subject03...
  Processing Subject04...
  Processing Subject05...
Not setting metadata
893 matching events found
No baseline correction applied
Not setting metadata
305 matching events found
No baseline correction applied

✅ Processing complete!
Total low-attention epochs from all subjects: 893
Total high-attention epochs from all subjects: 305


BIASED MODEL

In [12]:
# Cell 4: Feature Extraction 
from scipy.signal import welch
import numpy as np
import pandas as pd

# --- Function to calculate the Theta/Beta Ratio (no changes here) ---
def get_theta_beta_ratio(epoch_data, sfreq):
    """Calculates the theta/beta power ratio for a single EEG epoch."""
    theta_band = (4, 8)
    beta_band = (13, 30)
    freqs, psd = welch(epoch_data, sfreq, nperseg=sfreq*2, noverlap=sfreq)
    theta_power = np.mean(psd[np.where((freqs >= theta_band[0]) & (freqs <= theta_band[1]))])
    beta_power = np.mean(psd[np.where((freqs >= beta_band[0]) & (freqs <= beta_band[1]))])
    if beta_power == 0: return 0
    return theta_power / beta_power

# --- Loop through all epochs and extract the feature ---

print("Extracting features from all epochs...")

sfreq = final_low_epochs.info['sfreq']

# --- THIS IS THE CORRECTED PART ---
# We directly use 'epoch.squeeze()' because 'epoch' is already the numpy array.
low_attention_features = [get_theta_beta_ratio(epoch.squeeze(), sfreq) for epoch in final_low_epochs]
high_attention_features = [get_theta_beta_ratio(epoch.squeeze(), sfreq) for epoch in final_high_epochs]

print("✅ Feature extraction complete!")

# --- VALIDATION STEP: Let's see if the feature works! ---
avg_low_attention_ratio = np.mean(low_attention_features)
avg_high_attention_ratio = np.mean(high_attention_features)

print(f"\nAverage Theta/Beta Ratio for LOW attention: {avg_low_attention_ratio:.4f}")
print(f"Average Theta/Beta Ratio for HIGH attention: {avg_high_attention_ratio:.4f}")

if avg_high_attention_ratio < avg_low_attention_ratio:
    print("\nSuccess! As expected, the ratio is lower for the high-attention state.")
else:
    print("\nNote: The average ratio for high attention was not lower, which is unexpected but can happen depending on the data.")

Extracting features from all epochs...
✅ Feature extraction complete!

Average Theta/Beta Ratio for LOW attention: 6.8142
Average Theta/Beta Ratio for HIGH attention: 8.4898

Note: The average ratio for high attention was not lower, which is unexpected but can happen depending on the data.


In [13]:
# Cell 5: Build and Train the Classifier- LOGISITIC REGRESSION

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np

# --- Prepare the data for training ---
# Assumes low_attention_features and high_attention_features are in memory

# Combine all features into one list
features = low_attention_features + high_attention_features
# Create corresponding labels: 0 for low attention, 1 for high attention
labels = [0] * len(low_attention_features) + [1] * len(high_attention_features)

# It's good practice to structure this data in a pandas DataFrame
df = pd.DataFrame({'ratio_feature': features, 'attention_label': labels})

# X contains our feature(s)
X = df[['ratio_feature']].values
# y contains our labels
y = df['attention_label']

# --- Split data into a training set and a testing set ---
# The model learns from the training set and we check its performance on the unseen testing set.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- Initialize and Train the Model ---
# We'll use a simple and effective Logistic Regression model
model = LogisticRegression()

print("Training the model...")
# Train the model on the training data
model.fit(X_train, y_train)
print("✅ Model training complete!")

# --- Evaluate the Model's Performance ---
# Use the trained model to make predictions on the test set
predictions = model.predict(X_test)

# Calculate the accuracy of the predictions
accuracy = accuracy_score(y_test, predictions)

print(f"\nClassifier Accuracy: {accuracy*100:.2f}%")
print("This score tells us how well the model learned to separate the two states.")

Training the model...
✅ Model training complete!

Classifier Accuracy: 74.17%
This score tells us how well the model learned to separate the two states.


In [14]:
# Cell 6: Save the Trained Model
import joblib

# The 'model' variable holds the classifier we trained in a previous step
# Ensure 'model' is defined before this cell is run.
joblib.dump(model, 'attention_model.joblib')

print("✅ Model saved as 'attention_model.joblib' in your current working directory.")

✅ Model saved as 'attention_model.joblib' in your current working directory.


UNDERSAMPLING

In [15]:
#Cell 7: Balancing the Dataset via Undersampling
import numpy as np

# Get the number of epochs in the smaller class (high-attention)
n_high_epochs = len(final_high_epochs)

# Get the indices of the larger class (low-attention)
n_low_epochs = len(final_low_epochs)
low_attention_indices = np.arange(n_low_epochs)

# Randomly choose indices from the low-attention class to match the high-attention class size
random_indices = np.random.choice(low_attention_indices, n_high_epochs, replace=False)

# Create the balanced dataset by selecting the random subset of low-attention epochs
balanced_low_epochs = final_low_epochs[random_indices]

print("✅ Data has been balanced via undersampling.")
print(f"Total low-attention epochs: {len(balanced_low_epochs)}")
print(f"Total high-attention epochs: {len(final_high_epochs)}")

✅ Data has been balanced via undersampling.
Total low-attention epochs: 305
Total high-attention epochs: 305


SINGLE CHANNEL POWER MODEL

In [16]:
# Cell 8: Modern Feature Extraction using compute_psd()
def extract_power_ratio_features(epochs):
    """Calculates the theta/beta power ratio for each epoch using the modern .compute_psd() method."""
    
    # Define frequency bands
    THETA_BAND = (4.0, 8.0)
    BETA_BAND = (13.0, 30.0)
    
    # Get the power spectral density (PSD) using the .compute_psd() method
    # This is the corrected line
    spectrum = epochs.compute_psd(method='welch', n_fft=256, n_per_seg=256, fmin=1.0, fmax=40.0, verbose=False)
    psds, freqs = spectrum.get_data(return_freqs=True)
    
    # Calculate the power in each band
    # The psds array has shape (n_epochs, n_channels, n_freqs), so we select the first channel [:, 0, :]
    theta_power = np.sum(psds[:, 0, (freqs >= THETA_BAND[0]) & (freqs <= THETA_BAND[1])], axis=1)
    beta_power = np.sum(psds[:, 0, (freqs >= BETA_BAND[0]) & (freqs <= BETA_BAND[1])], axis=1)
    
    # Calculate the ratio
    power_ratio = theta_power / beta_power
    
    # Reshape for the model (n_epochs, n_features)
    return power_ratio.reshape(-1, 1)

# Extract features for both classes using the corrected function
low_attention_features = extract_power_ratio_features(balanced_low_epochs)
high_attention_features = extract_power_ratio_features(final_high_epochs)

print("✅ Features extracted successfully with the corrected method.")
print(f"Shape of low-attention features: {low_attention_features.shape}")
print(f"Shape of high-attention features: {high_attention_features.shape}")

✅ Features extracted successfully with the corrected method.
Shape of low-attention features: (305, 1)
Shape of high-attention features: (305, 1)


In [17]:
# Cell 9: Data Preparation — Scaling and Splitting for Model Training

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Combine features into a single array X
X = np.vstack([low_attention_features, high_attention_features])

# Create corresponding labels y
n_samples_per_class = low_attention_features.shape[0]
y = np.array([0] * n_samples_per_class + [1] * n_samples_per_class)

# Split the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features for better model performance
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✅ Data prepared and split for training.")
print(f"Training data shape: {X_train_scaled.shape}")
print(f"Testing data shape: {X_test_scaled.shape}")


✅ Data prepared and split for training.
Training data shape: (488, 1)
Testing data shape: (122, 1)


In [18]:
#Cell 10: Training & Evaluating the Logistic Regression Classifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train the model
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_scaled)

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=['Low Attention', 'High Attention'])

print("\n✅ Model training and evaluation complete.")
print(f"\nModel Accuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(report)


✅ Model training and evaluation complete.

Model Accuracy: 0.59

Classification Report:
                precision    recall  f1-score   support

 Low Attention       0.57      0.74      0.64        61
High Attention       0.63      0.44      0.52        61

      accuracy                           0.59       122
     macro avg       0.60      0.59      0.58       122
  weighted avg       0.60      0.59      0.58       122



MODIFIED SINGLE CHANNEL POWER MODEL

In [19]:
# Cell 11: EEG Attention Classification using Random Forest

# Import necessary libraries for data handling, signal processing, and machine learning.
import mne
import os
import glob
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 2. DATA PREPROCESSING AND EPOCHING
def preprocess_and_epoch_file(file_path, channel_name='EEG Fz'):
    """Loads an EDF file, selects a channel, filters, and creates epochs."""
    if not os.path.exists(file_path):
        return None
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    raw.pick(channel_name)
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    epochs = mne.make_fixed_length_epochs(raw, duration=2, overlap=1, preload=True, verbose=False)
    return epochs

print("INFO: Starting data loading and preprocessing...")
data_folder = "eeg-during-mental-arithmetic-tasks-1.0.0"
subjects_to_process = [f"Subject{i:02d}" for i in range(0, 6)]

all_low_attention_epochs = []
all_high_attention_epochs = []

for subject in subjects_to_process:
    low_attention_file = os.path.join(data_folder, f"{subject}_1.edf")
    high_attention_file = os.path.join(data_folder, f"{subject}_2.edf")
    
    low_epochs = preprocess_and_epoch_file(low_attention_file)
    if low_epochs:
        all_low_attention_epochs.append(low_epochs)
        
    high_epochs = preprocess_and_epoch_file(high_attention_file)
    if high_epochs:
        all_high_attention_epochs.append(high_epochs)

final_low_epochs = mne.concatenate_epochs(all_low_attention_epochs)
final_high_epochs = mne.concatenate_epochs(all_high_attention_epochs)
print("✅ Preprocessing complete.")

# 3. DATA BALANCING (UNDERSAMPLING)
print("\nINFO: Balancing dataset via undersampling...")
n_high_epochs = len(final_high_epochs)
n_low_epochs = len(final_low_epochs)

random_indices = np.random.choice(np.arange(n_low_epochs), n_high_epochs, replace=False)
balanced_low_epochs = final_low_epochs[random_indices]
print(f"✅ Data balanced. Using {len(balanced_low_epochs)} epochs per class.")

# 4. ENHANCED FEATURE EXTRACTION
def extract_enhanced_features(epochs):
    """Calculates power in multiple bands and the theta/beta ratio."""
    THETA_BAND = (4.0, 8.0)
    ALPHA_BAND = (8.0, 13.0)
    BETA_BAND = (13.0, 30.0)
    
    spectrum = epochs.compute_psd(method='welch', n_fft=256, n_per_seg=256, fmin=1.0, fmax=40.0, verbose=False)
    psds, freqs = spectrum.get_data(return_freqs=True)
    
    theta_power = np.sum(psds[:, 0, (freqs >= THETA_BAND[0]) & (freqs <= THETA_BAND[1])], axis=1)
    alpha_power = np.sum(psds[:, 0, (freqs >= ALPHA_BAND[0]) & (freqs <= ALPHA_BAND[1])], axis=1)
    beta_power = np.sum(psds[:, 0, (freqs >= BETA_BAND[0]) & (freqs <= BETA_BAND[1])], axis=1)
    
    theta_beta_ratio = np.divide(theta_power, beta_power, out=np.zeros_like(theta_power), where=beta_power!=0)
    
    features = np.vstack([theta_power, alpha_power, beta_power, theta_beta_ratio]).T
    return features

print("\nINFO: Extracting enhanced features from balanced epochs...")
low_attention_features = extract_enhanced_features(balanced_low_epochs)
high_attention_features = extract_enhanced_features(final_high_epochs)
print("✅ Feature extraction complete.")

# 5. MODEL TRAINING AND EVALUATION
# Combine features and create corresponding labels.
X = np.vstack((low_attention_features, high_attention_features))
y = np.array([0] * len(low_attention_features) + [1] * len(high_attention_features))

# Split the data into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Scale features for better model performance.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize, train, and evaluate the RandomForest model.
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print("\n" + "="*20 + " FINAL MODEL RESULTS " + "="*20)
print(f"Final Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Low-Attention', 'High-Attention']))
print("="*53 + "\n")

INFO: Starting data loading and preprocessing...
Not setting metadata
1074 matching events found
No baseline correction applied
Not setting metadata
366 matching events found
No baseline correction applied
✅ Preprocessing complete.

INFO: Balancing dataset via undersampling...
✅ Data balanced. Using 366 epochs per class.

INFO: Extracting enhanced features from balanced epochs...
✅ Feature extraction complete.

==================== FINAL MODEL RESULTS ====================
Final Model Accuracy: 57.38%

Classification Report:
                precision    recall  f1-score   support

 Low-Attention       0.58      0.57      0.57        92
High-Attention       0.57      0.58      0.58        91

      accuracy                           0.57       183
     macro avg       0.57      0.57      0.57       183
  weighted avg       0.57      0.57      0.57       183




In [20]:
#Cell 12: Subject-Wise EEG Attention Classification Pipeline
# 1. IMPORTS
import mne
import os
import glob
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings

# Suppress some of the verbose MNE warnings for cleaner output
warnings.filterwarnings('ignore', category=RuntimeWarning, module='mne')

# 2. CONFIGURATION
DATA_FOLDER = "eeg-during-mental-arithmetic-tasks-1.0.0"
SUBJECTS_TO_PROCESS = [f"Subject{i:02d}" for i in range(0, 6)] # Subjects 00 to 05
CHANNEL_TO_USE = 'EEG Fz' # The frontal channel we selected

# 3. HELPER FUNCTIONS
def preprocess_and_epoch_file(file_path, channel_name):
    """Loads an EDF file, selects a channel, filters, and creates epochs."""
    if not os.path.exists(file_path):
        return None
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    raw.pick(channel_name)
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    epochs = mne.make_fixed_length_epochs(raw, duration=2, overlap=1, preload=True, verbose=False)
    return epochs

def extract_enhanced_features(epochs):
    """Calculates power in multiple bands and the theta/beta ratio."""
    THETA_BAND = (4.0, 8.0)
    ALPHA_BAND = (8.0, 13.0)
    BETA_BAND = (13.0, 30.0)
    
    spectrum = epochs.compute_psd(method='welch', n_fft=256, n_per_seg=256, fmin=1.0, fmax=40.0, verbose=False)
    psds, freqs = spectrum.get_data(return_freqs=True)
    
    theta_power = np.sum(psds[:, 0, (freqs >= THETA_BAND[0]) & (freqs <= THETA_BAND[1])], axis=1)
    alpha_power = np.sum(psds[:, 0, (freqs >= ALPHA_BAND[0]) & (freqs <= ALPHA_BAND[1])], axis=1)
    beta_power = np.sum(psds[:, 0, (freqs >= BETA_BAND[0]) & (freqs <= BETA_BAND[1])], axis=1)
    
    theta_beta_ratio = np.divide(theta_power, beta_power, out=np.zeros_like(theta_power), where=beta_power!=0)
    
    features = np.vstack([theta_power, alpha_power, beta_power, theta_beta_ratio]).T
    return features

# 4. MAIN PROCESSING LOOP (SUBJECT-SPECIFIC)
all_subject_accuracies = []

print("INFO: Starting subject-specific model training...")
for subject in SUBJECTS_TO_PROCESS:
    print(f"\n--- Processing {subject} ---")
    
    # 4a. Load data for the current subject
    low_attention_file = os.path.join(DATA_FOLDER, f"{subject}_1.edf")
    high_attention_file = os.path.join(DATA_FOLDER, f"{subject}_2.edf")
    
    low_epochs = preprocess_and_epoch_file(low_attention_file, CHANNEL_TO_USE)
    high_epochs = preprocess_and_epoch_file(high_attention_file, CHANNEL_TO_USE)
    
    if low_epochs is None or high_epochs is None:
        print(f"Warning: Could not find data for {subject}. Skipping.")
        continue
        
    # 4b. Balance data for the current subject
    n_high = len(high_epochs)
    n_low = len(low_epochs)
    n_balanced = min(n_high, n_low)
    
    low_epochs.crop(tmax=(n_balanced * 2) - 1) # Trim to match size
    high_epochs.crop(tmax=(n_balanced * 2) - 1)
    
    # 4c. Extract features for the current subject
    low_features = extract_enhanced_features(low_epochs)
    high_features = extract_enhanced_features(high_epochs)
    
    # 4d. Create dataset and train model for the current subject
    X = np.vstack((low_features, high_features))
    y = np.array([0] * len(low_features) + [1] * len(high_features))
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    all_subject_accuracies.append(accuracy)
    
    print(f"✅ Accuracy for {subject}: {accuracy * 100:.2f}%")

# 5. FINAL RESULTS
if all_subject_accuracies:
    mean_accuracy = np.mean(all_subject_accuracies)
    print("\n" + "="*20 + " OVERALL RESULTS " + "="*20)
    print(f"Average Accuracy Across All Subjects: {mean_accuracy * 100:.2f}%")
    print("="*55 + "\n")
else:
    print("\nError: No subjects were processed. Please check file paths and subject list.")

INFO: Starting subject-specific model training...

--- Processing Subject00 ---
✅ Accuracy for Subject00: 70.49%

--- Processing Subject01 ---
✅ Accuracy for Subject01: 78.69%

--- Processing Subject02 ---
✅ Accuracy for Subject02: 72.13%

--- Processing Subject03 ---
✅ Accuracy for Subject03: 91.80%

--- Processing Subject04 ---
✅ Accuracy for Subject04: 79.31%

--- Processing Subject05 ---
✅ Accuracy for Subject05: 77.05%

==================== OVERALL RESULTS ====================
Average Accuracy Across All Subjects: 78.25%



MULTI-CHANNEL CONNECTIVITY MODEL

In [21]:
#CELL 13: Multi-Channel EEG Processing 
import mne
import os
import numpy as np

def preprocess_multi_channel(file_path):
    """Loads and filters an EEG file, keeping all available EEG channels."""
    if not os.path.exists(file_path):
        print(f"Warning: File not found at {file_path}")
        return None
    
    # Load the data and preload it into memory
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    
    # --- THIS IS THE KEY CHANGE ---
    # Instead of picking one channel, we select all channels of type 'eeg'
    raw.pick_types(eeg=True, exclude='bads') # This selects all EEG channels
    
    # Apply the same filter as before
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    
    # Create epochs
    epochs = mne.make_fixed_length_epochs(raw, duration=2, overlap=1, preload=True, verbose=False)
    
    return epochs

# --- Run the multi-channel preprocessing ---
# (This part is similar to before, but uses the new function)

subjects_to_process = [f"Subject{i:02d}" for i in range(1, 6)]
data_folder = "eeg-during-mental-arithmetic-tasks-1.0.0"

all_low_attention_epochs_mc = [] # mc for multi-channel
all_high_attention_epochs_mc = []

print("Starting multi-channel data processing...")
for subject in subjects_to_process:
    low_attention_file = os.path.join(data_folder, f"{subject}_1.edf")
    high_attention_file = os.path.join(data_folder, f"{subject}_2.edf")

    low_epochs = preprocess_multi_channel(low_attention_file)
    if low_epochs:
        all_low_attention_epochs_mc.append(low_epochs)

    high_epochs = preprocess_multi_channel(high_attention_file)
    if high_epochs:
        all_high_attention_epochs_mc.append(high_epochs)

# Combine epochs from all subjects
final_low_epochs_mc = mne.concatenate_epochs(all_low_attention_epochs_mc)
final_high_epochs_mc = mne.concatenate_epochs(all_high_attention_epochs_mc)

# --- Balance the new multi-channel data ---
n_high_mc = len(final_high_epochs_mc)
n_low_mc = len(final_low_epochs_mc)
random_indices_mc = np.random.choice(n_low_mc, n_high_mc, replace=False)

balanced_low_epochs_mc = final_low_epochs_mc[random_indices_mc]

print("\n✅ Multi-channel data processed and balanced.")
print(f"Number of channels loaded: {len(balanced_low_epochs_mc.ch_names)}")
print(f"Total low-attention epochs: {len(balanced_low_epochs_mc)}")
print(f"Total high-attention epochs: {len(final_high_epochs_mc)}")

Starting multi-channel data processing...
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Not setting metadata
893 matching events found
No baseline correction applied
Not setting metadata
305 matching events found
No baseline correction applied

✅ Multi-channel data processed 

In [22]:
#	CELL 14- Multi-Channel EEG Preprocessing – Finalized
import mne
import os
import numpy as np

def preprocess_multi_channel(file_path):
    """Loads and filters an EEG file, keeping all available EEG channels."""
    if not os.path.exists(file_path):
        print(f"Warning: File not found at {file_path}")
        return None
    
    # Load the data and preload it into memory
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    
    # --- THIS IS THE CORRECTED LINE ---
    # The channel type 'eeg' is passed directly without the 'types=' keyword.
    raw.pick('eeg', exclude='bads')
    
    # Apply the same filter as before
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    
    # Create epochs
    epochs = mne.make_fixed_length_epochs(raw, duration=2, overlap=1, preload=True, verbose=False)
    
    return epochs

# --- Run the multi-channel preprocessing ---
subjects_to_process = [f"Subject{i:02d}" for i in range(1, 6)]
data_folder = "eeg-during-mental-arithmetic-tasks-1.0.0"

all_low_attention_epochs_mc = [] # mc for multi-channel
all_high_attention_epochs_mc = []

print("Starting multi-channel data processing...")
for subject in subjects_to_process:
    low_attention_file = os.path.join(data_folder, f"{subject}_1.edf")
    high_attention_file = os.path.join(data_folder, f"{subject}_2.edf")

    low_epochs = preprocess_multi_channel(low_attention_file)
    if low_epochs:
        all_low_attention_epochs_mc.append(low_epochs)

    high_epochs = preprocess_multi_channel(high_attention_file)
    if high_epochs:
        all_high_attention_epochs_mc.append(high_epochs)

# Combine epochs from all subjects
final_low_epochs_mc = mne.concatenate_epochs(all_low_attention_epochs_mc)
final_high_attention_epochs_mc = mne.concatenate_epochs(all_high_attention_epochs_mc)

print("\n✅ Step 1 Complete: Multi-channel data processed.")

Starting multi-channel data processing...
Not setting metadata
893 matching events found
No baseline correction applied
Not setting metadata
305 matching events found
No baseline correction applied

✅ Step 1 Complete: Multi-channel data processed.


In [23]:
#CELL 15 – Balancing Multi-Channel EEG Data
# Balance the new multi-channel data
n_high_mc = len(final_high_epochs_mc)
n_low_mc = len(final_low_epochs_mc)
random_indices_mc = np.random.choice(n_low_mc, n_high_mc, replace=False)

balanced_low_epochs_mc = final_low_epochs_mc[random_indices_mc]

print("✅ Step 2 Complete: Multi-channel data balanced.")
print(f"Number of channels loaded: {len(balanced_low_epochs_mc.ch_names)}")
print(f"Total low-attention epochs: {len(balanced_low_epochs_mc)}")
print(f"Total high-attention epochs: {len(final_high_epochs_mc)}")

✅ Step 2 Complete: Multi-channel data balanced.
Number of channels loaded: 21
Total low-attention epochs: 305
Total high-attention epochs: 305


In [24]:
#CELL 16 – Manual PLV Feature Extraction & Classification

# --- Imports ---
import mne
import os
import numpy as np
from scipy.signal import hilbert
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# --- Step 1: Define Preprocessing Function and Load Data ---

def preprocess_multi_channel(file_path):
    """Loads and filters an EEG file, keeping all available EEG channels."""
    if not os.path.exists(file_path):
        return None
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    raw.pick('eeg', exclude='bads')
    raw.filter(1., 40., fir_design='firwin', verbose=False)
    epochs = mne.make_fixed_length_epochs(raw, duration=2, overlap=1, preload=True, verbose=False)
    return epochs

# --- Run the multi-channel preprocessing ---
print("Starting multi-channel data processing...")
# !!! IMPORTANT: Update this path to your data's location !!!
data_folder = "eeg-during-mental-arithmetic-tasks-1.0.0" 

subjects_to_process = [f"Subject{i:02d}" for i in range(1, 6)]

all_low_attention_epochs_mc = []
all_high_attention_epochs_mc = []

for subject in subjects_to_process:
    low_attention_file = os.path.join(data_folder, f"{subject}_1.edf")
    high_attention_file = os.path.join(data_folder, f"{subject}_2.edf")
    low_epochs = preprocess_multi_channel(low_attention_file)
    if low_epochs:
        all_low_attention_epochs_mc.append(low_epochs)
    high_epochs = preprocess_multi_channel(high_attention_file)
    if high_epochs:
        all_high_attention_epochs_mc.append(high_epochs)

final_low_epochs_mc = mne.concatenate_epochs(all_low_attention_epochs_mc)
final_high_attention_epochs_mc = mne.concatenate_epochs(all_high_attention_epochs_mc)
print("✅ Data loading complete.")

# --- Step 2: Balance the Data ---

n_high_mc = len(final_high_attention_epochs_mc)
n_low_mc = len(final_low_epochs_mc)
random_indices_mc = np.random.choice(n_low_mc, n_high_mc, replace=False)
balanced_low_epochs_mc = final_low_epochs_mc[random_indices_mc]
print("✅ Data balancing complete.")

# --- Step 3: Define MANUAL Feature Extraction and Extract Features ---

def extract_plv_features_manual(epochs):
    """
    Manually calculates PLV in the alpha band without using mne.connectivity.
    """
    # Get data as a NumPy array: (n_epochs, n_channels, n_samples)
    data = epochs.get_data(picks='eeg')
    sfreq = epochs.info['sfreq']
    n_epochs, n_channels, n_samples = data.shape
    
    # Filter data in the alpha band (8-13 Hz)
    data_alpha = mne.filter.filter_data(data, sfreq, 8, 13, verbose=False)
    
    # Get analytic signal and instantaneous phase using the Hilbert transform
    analytic_signal = hilbert(data_alpha, axis=2)
    phase_data = np.angle(analytic_signal)
    
    # Compute PLV for each epoch
    plv_features = []
    for epoch_idx in range(n_epochs):
        # Calculate phase differences for all channel pairs
        plv_matrix = np.zeros((n_channels, n_channels))
        for i in range(n_channels):
            for j in range(i, n_channels):
                if i == j:
                    continue
                phase_diff = phase_data[epoch_idx, i, :] - phase_data[epoch_idx, j, :]
                # Compute PLV value
                plv = np.abs(np.mean(np.exp(1j * phase_diff)))
                plv_matrix[i, j] = plv
                plv_matrix[j, i] = plv
        
        # Flatten the upper triangle of the matrix to create a feature vector
        triu_indices = np.triu_indices(n_channels, k=1)
        plv_features.append(plv_matrix[triu_indices])

    return np.array(plv_features)

# --- Run feature extraction ---
print("Extracting PLV features manually... (This may take longer)")
low_attention_plv_features = extract_plv_features_manual(balanced_low_epochs_mc)
high_attention_plv_features = extract_plv_features_manual(final_high_epochs_mc)
print("✅ Manual PLV features extracted.")

# --- Step 4: Train and Evaluate the Model ---
print("Preparing data and training model...")

# Prepare data
X_plv = np.vstack([low_attention_plv_features, high_attention_plv_features])
y_plv = np.array([0] * len(low_attention_plv_features) + [1] * len(high_attention_plv_features))

# Split data
X_train_plv, X_test_plv, y_train_plv, y_test_plv = train_test_split(
    X_plv, y_plv, test_size=0.2, random_state=42, stratify=y_plv
)

# Scale features
scaler_plv = StandardScaler()
X_train_plv_scaled = scaler_plv.fit_transform(X_train_plv)
X_test_plv_scaled = scaler_plv.transform(X_test_plv)

# Train and evaluate
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_plv_scaled, y_train_plv)
y_pred_plv = rf_model.predict(X_test_plv_scaled)

accuracy_plv = accuracy_score(y_test_plv, y_pred_plv)
report_plv = classification_report(y_test_plv, y_pred_plv, target_names=['Low Attention', 'High Attention'])

# --- Final Output ---
print("\n--- Final Results (Manual PLV) ---")
print(f"Model Accuracy with PLV features: {accuracy_plv:.2f}")
print("\nConnectivity Model Classification Report:")
print(report_plv)

Starting multi-channel data processing...
Not setting metadata
893 matching events found
No baseline correction applied
Not setting metadata
305 matching events found
No baseline correction applied
✅ Data loading complete.
✅ Data balancing complete.
Extracting PLV features manually... (This may take longer)
✅ Manual PLV features extracted.
Preparing data and training model...

--- Final Results (Manual PLV) ---
Model Accuracy with PLV features: 0.85

Connectivity Model Classification Report:
                precision    recall  f1-score   support

 Low Attention       0.85      0.85      0.85        61
High Attention       0.85      0.85      0.85        61

      accuracy                           0.85       122
     macro avg       0.85      0.85      0.85       122
  weighted avg       0.85      0.85      0.85       122



In [25]:
#CELL 17- Saving the Connectivity Model 
import joblib

# 'rf_model' and 'scaler_plv' should now exist from the previous cell
joblib.dump(rf_model, 'connectivity_model.joblib')
joblib.dump(scaler_plv, 'connectivity_scaler.joblib')

print("✅ Model and scaler saved successfully.")

✅ Model and scaler saved successfully.
